# ⚽ FootballFlow — Silver to InfluxDB (Live Match Feed)

## Overview
This notebook reads clean events from the **Silver Layer** as a continuous stream
and writes them to **InfluxDB** in real-time.

InfluxDB is a time-series database optimized for live data —
Grafana reads from it to display the **Live Match Dashboard**.

```
Silver Parquet (clean, typed)
        ↓
  Spark Structured Streaming
        ↓
     InfluxDB
        ↓
  Grafana — Live Match Page
```

**Input Path:**  `~/footballflow/silver/match_events/`  
**Output:**      InfluxDB bucket `match_events`  
**Trigger:**     Every 10 seconds

---

## InfluxDB Connection Info
| Parameter | Value |
|-----------|-------|
| URL | http://localhost:8086 |
| Org | footballflow |
| Bucket | match_events |
| Token | a-4Jo-oJge1e-GeV3DDZCOP_iVQTdTqJk3L01iGk94DYdPoRsyWzVCyHcASacw_PDePa-xIic5JjfhTJ2Jfh6Q== |


---
## Cell 1 — Imports


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType,
    TimestampType, DateType
)
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
from datetime import datetime

---
## Cell 2 — InfluxDB Config

Set your InfluxDB API Token here.


In [2]:
# ── InfluxDB Connection ───────────────────────────────────────────────────────
INFLUX_URL    = "http://host.docker.internal:8086"
INFLUX_TOKEN  = "a-4Jo-oJge1e-GeV3DDZCOP_iVQTdTqJk3L01iGk94DYdPoRsyWzVCyHcASacw_PDePa-xIic5JjfhTJ2Jfh6Q=="  # ← Replace with your token
INFLUX_ORG    = "footballflow"
INFLUX_BUCKET = "match_events"

# Verify the connection is working
client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
health = client.health()
print(f"✅ InfluxDB connection : {health.status}")
print(f"   URL                : {INFLUX_URL}")
print(f"   Org                : {INFLUX_ORG}")
print(f"   Bucket             : {INFLUX_BUCKET}")
client.close()


✅ InfluxDB connection : pass
   URL                : http://host.docker.internal:8086
   Org                : footballflow
   Bucket             : match_events


---
## Cell 3 — SparkSession

In [3]:
jar_dir = os.path.expanduser("~/kafka-jars")
jars = ",".join([
    f"{jar_dir}/spark-sql-kafka-0-10_2.12-3.5.3.jar",
    f"{jar_dir}/kafka-clients-3.4.1.jar",
    f"{jar_dir}/spark-token-provider-kafka-0-10_2.12-3.5.3.jar",
    f"{jar_dir}/commons-pool2-2.11.1.jar"
])

spark = SparkSession.builder \
    .appName("FootballFlow-InfluxDB") \
    .config("spark.jars", jars) \
    .config("spark.sql.shuffle.partitions", "3") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")

Spark version : 3.5.3
App name      : FootballFlow-InfluxDB


---
## Cell 4 — Paths

In [4]:
silver_path     = os.path.expanduser("~/footballflow/silver/match_events/")
checkpoint_path = os.path.expanduser("~/footballflow/checkpoints/silver_to_influxdb/")

print(f"Silver     : {silver_path}")
print(f"Checkpoint : {checkpoint_path}")

Silver     : /home/jovyan/footballflow/silver/match_events/
Checkpoint : /home/jovyan/footballflow/checkpoints/silver_to_influxdb/


---
## Cell 5 — Silver Schema

This schema matches the enriched Silver dataset —
all event data is self-contained in each row, with no need for external lookup dictionaries.


In [5]:
silver_schema = StructType([
    # ── Event ──────────────────────────────────────────────────────
    StructField("game_event_id",              StringType(),    True),
    StructField("game_date",                  DateType(),      True),
    StructField("game_id",                    IntegerType(),   True),
    StructField("minute",                     IntegerType(),   True),
    StructField("event_type",                 StringType(),    True),
    StructField("description",                StringType(),    True),
    StructField("season_count",               IntegerType(),   True),
    StructField("penalty_result",             StringType(),    True),
    StructField("player_in_id",               StringType(),    True),
    StructField("player_assist_id",           StringType(),    True),

    # ── Player ─────────────────────────────────────────────────────
    StructField("player_id",                  IntegerType(),   True),
    StructField("player_name",                StringType(),    True),
    StructField("player_position",            StringType(),    True),
    StructField("player_sub_position",        StringType(),    True),
    StructField("player_foot",                StringType(),    True),
    StructField("player_nationality",         StringType(),    True),
    StructField("player_height_cm",           FloatType(),     True),
    StructField("player_market_value",        FloatType(),     True),

    # ── Club ───────────────────────────────────────────────────────
    StructField("club_id",                    IntegerType(),   True),
    StructField("club_name",                  StringType(),    True),
    StructField("coach_name",                 StringType(),    True),
    StructField("lineup_type",                StringType(),    True),
    StructField("lineup_position",            StringType(),    True),
    StructField("shirt_number",               IntegerType(),   True),
    StructField("team_captain",               IntegerType(),   True),

    # ── Match Stats ────────────────────────────────────────────────
    StructField("club_goals_scored",          IntegerType(),   True),
    StructField("club_goals_conceded",        IntegerType(),   True),
    StructField("club_league_position",       FloatType(),     True),
    StructField("opponent_league_position",   FloatType(),     True),
    StructField("hosting",                    StringType(),    True),
    StructField("is_win",                     IntegerType(),   True),

    # ── Match Info ─────────────────────────────────────────────────
    StructField("home_club_id",               StringType(),    True),
    StructField("away_club_id",               StringType(),    True),
    StructField("home_club_name",             StringType(),    True),
    StructField("away_club_name",             StringType(),    True),
    StructField("home_club_goals",            IntegerType(),   True),
    StructField("away_club_goals",            IntegerType(),   True),
    StructField("home_club_formation",        StringType(),    True),
    StructField("away_club_formation",        StringType(),    True),
    StructField("stadium",                    StringType(),    True),
    StructField("attendance",                 FloatType(),     True),
    StructField("referee",                    StringType(),    True),
    StructField("season",                     IntegerType(),   True),
    StructField("round",                      StringType(),    True),

    # ── Competition ────────────────────────────────────────────────
    StructField("competition_name",           StringType(),    True),
    StructField("competition_type",           StringType(),    True),
    StructField("country_name",               StringType(),    True),
    StructField("confederation",              StringType(),    True),

    # ── Metadata ───────────────────────────────────────────────────
    StructField("_bronze_ingested_at",        TimestampType(), True),
    StructField("_source_topic",              StringType(),    True),
    StructField("_pipeline",                  StringType(),    True),
    StructField("_silver_processed_at",       TimestampType(), True),
])

print(f"✅ Silver schema defined — {len(silver_schema.fields)} columns")

✅ Silver schema defined — 52 columns


---
## Cell 6 — Write to InfluxDB Function

This function writes each micro-batch from the Silver Layer to InfluxDB.

### InfluxDB Data Model:
- **Measurement:** `event_type` (goal / card / substitution / other)
- **Tags:** Fields used for filtering (game_id, club_name, event_type, competition_name)
- **Fields:** Event payload (minute, player_name, description, goals, stadium...)
- **Timestamp:** Actual write time (UTC)


In [6]:
def write_to_influxdb(batch_df, batch_id):
    rows = batch_df.collect()
    if not rows:
        print(f"Batch {batch_id} — no new rows")
        return

    client    = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
    write_api = client.write_api(write_options=SYNCHRONOUS)

    points = []
    for row in rows:
        event_type = row["event_type"] or "other"

        point = (
            Point(event_type)
            # Tags — used for filtering and grouping in Grafana
            .tag("game_id",          str(row["game_id"]))
            .tag("club_name",        str(row["club_name"]        or ""))
            .tag("event_type",       event_type)
            .tag("competition_name", str(row["competition_name"] or ""))
            # Fields — event payload values
            .field("minute",             int(row["minute"])             if row["minute"]             else 0)
            .field("player_id",          int(row["player_id"])          if row["player_id"]          else 0)
            .field("player_name",        str(row["player_name"]         or "Unknown"))
            .field("description",        str(row["description"]         or ""))
            .field("game_event_id",      str(row["game_event_id"]       or ""))
            .field("home_club_goals",    int(row["home_club_goals"])    if row["home_club_goals"]    else 0)
            .field("away_club_goals",    int(row["away_club_goals"])    if row["away_club_goals"]    else 0)
            .field("stadium",            str(row["stadium"]             or ""))
            .field("season_count",       int(row["season_count"])       if row["season_count"]       else 0)
            .field("player_nationality", str(row["player_nationality"]  or ""))
            .field("sub_position",       str(row["player_sub_position"] or ""))
            .field("referee",            str(row["referee"]             or ""))
            .field("attendance",         float(row["attendance"])       if row["attendance"]         else 0)
            .field("round",              str(row["round"]               or ""))
            .field("coach_name",         str(row["coach_name"]          or ""))
            .field("country_name",       str(row["country_name"]        or ""))
            .field("season",             int(row["season"])             if row["season"]             else 0)
            .field("game_date",          str(row["game_date"])          if row["game_date"] else "")
            .field("penalty_result",     str(row["penalty_result"]      or ""))
            .time(datetime.utcnow(), WritePrecision.NS)
        )
        points.append(point)

    write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=points)
    client.close()
    print(f"✅ Batch {batch_id} — wrote {len(points):,} points to InfluxDB")

print("✅ write_to_influxdb function defined")


✅ write_to_influxdb function defined


---
## Cell 7 — Read Silver Stream + Write to InfluxDB

Read from the Silver Layer as a stream and write to InfluxDB every 10 seconds.

**10-second trigger** — faster than Bronze/Silver (30 seconds) to keep the live dashboard as responsive as possible.


In [7]:
# Read Silver Parquet as a streaming source
silver_stream = spark.readStream \
    .schema(silver_schema) \
    .parquet(silver_path)

# Write to InfluxDB using foreachBatch
influx_query = silver_stream.writeStream \
    .outputMode("append") \
    .foreachBatch(write_to_influxdb) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Silver → InfluxDB Stream started!")
print(f"   Trigger    : every 10 seconds")
print(f"   Checkpoint : {checkpoint_path}")
print(f"   Status     : {influx_query.status}")


✅ Silver → InfluxDB Stream started!
   Trigger    : every 10 seconds
   Checkpoint : /home/jovyan/footballflow/checkpoints/silver_to_influxdb/
   Status     : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
✅ Batch 6 — wrote 1 points to InfluxDB
✅ Batch 7 — wrote 2 points to InfluxDB
✅ Batch 8 — wrote 5 points to InfluxDB
✅ Batch 9 — wrote 2 points to InfluxDB
✅ Batch 10 — wrote 3 points to InfluxDB


---
## Cell 8 — Monitor Stream

Run this cell to confirm that data is reaching InfluxDB.


In [8]:
import time
time.sleep(15)  # Wait for the first trigger to complete

print(f"🔄 Stream active  : {influx_query.isActive}")
print(f"📊 Last progress  : {influx_query.lastProgress}")

# Verify data in InfluxDB
client    = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
query_api = client.query_api()

query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -2h)
  |> filter(fn: (r) => r["_field"] == "minute" or r["_field"] == "player_name" or r["_field"] == "club_name")
  |> pivot(rowKey: ["_time", "game_id"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "_measurement", "game_id", "club_name", "player_name", "minute"])
  |> sort(columns: ["minute"], desc: false)
'''

tables = query_api.query(query)
total  = sum(len(t.records) for t in tables)
print(f"\n📊 Records in InfluxDB : {total}")

for table in tables:
    for record in table.records:
        print(f"   {record.get_measurement():<15} | minute={record.values.get('minute')} | player={record.values.get('player_name')} | club={record.values.get('club_name')}")

client.close()


🔄 Stream active  : True
📊 Last progress  : {'id': 'cf4e433f-4ed6-4038-87b1-024815380b6b', 'runId': '2b731189-0ebe-47ac-b2c4-fcb7a2090986', 'name': None, 'timestamp': '2026-06-12T21:12:52.082Z', 'batchId': 6, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 32, 'triggerExecution': 147}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[file:/home/jovyan/footballflow/silver/match_events]', 'startOffset': {'logOffset': 5}, 'endOffset': {'logOffset': 5}, 'latestOffset': None, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0}], 'sink': {'description': 'ForeachBatchSink', 'numOutputRows': -1}}

📊 Records in InfluxDB : 0


In [ ]:
influx_query.exception()

---
## Cell 9 — Stop Stream

Run only when you are done monitoring the stream.


In [ ]:
# Stop all active streaming queries
for q in spark.streams.active:
    q.stop()
    print(f"⛔ Stopped: {q.name}")

print(f"Active streams remaining: {len(spark.streams.active)}")


In [1]:
# Utility — Clear Silver files and all related checkpoints (use with caution)
rm -rf ~/footballflow/silver/match_events/* ~/footballflow/checkpoints/silver_to_influxdb/* ~/footballflow/checkpoints/silver_match_events/*


In [2]:
# Utility — Clear only the InfluxDB sink checkpoint
rm -rf ~/footballflow/checkpoints/silver_to_influxdb/*


In [3]:
rm -rf ~/footballflow/checkpoints/silver_to_influxdb/*


In [10]:
print(influx_query.exception())


An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/usr/local/spark/python/pyspark/sql/types.py", line 2378, in __getitem__
    idx = self.__fields__.index(item)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: 'event_date' is not in list

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_13425/1334621309.py", line 37, in write_to_influxdb
    .field("event_date",         str(row["event_date"])         if row["even